# Сравнение нескольких Word2Vec-моделей (авторские пространства)

Этот ноутбук реализует многоуровневое сравнение **word2vec-моделей разных авторов** и сопоставление с общеязыковой моделью **Ruscorpora** (как в тексте статьи).

## Общий принцип
- сравнение выполняется **только на пересечении словарей** (общий набор токенов `lemma_POS`)
- результаты сохраняются в `results/compare_w2v/` (таблицы + изображения)

## Методы сравнения (как в статье)
1) **Orthogonal Procrustes** (выровнять пары моделей) → затем считаем:
- **cosine displacement**: \(1-\cos(\hat{x}_w, y_w)\)
- **LND (Local Neighborhood Distance)**: насколько изменилась локальная геометрия вокруг слова (через соседей)

2) **Gromov–Wasserstein distance** (POT/ot): сравнение моделей как метрических структур без явного выравнивания координат

3) **Сравнение соседей слова** (без выравнивания):
- **Jaccard overlap** по множествам top-k соседей
- **RBO (Rank-Biased Overlap)** по *ранжированным* спискам соседей

## Воспроизводимость
- внутриавторские «replicates» (разные `seed`) используются для проверки устойчивости различий: **межавторские расстояния должны быть выше внутриавторских** (Приложение А).


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import KeyedVectors

BASE = Path.cwd()
MODELS_DIR = BASE / "models"
OUT_DIR = BASE / "results" / "compare_w2v"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

BASE, MODELS_DIR, OUT_DIR


In [ ]:
# Укажите модели для сравнения (пути к .kv, которые вы сохраняли в обучении)
# Тестовый список: только модели из models/w2v_authors/old
MODEL_PATHS = [
    BASE / "models" / "w2v_authors" / "chekhov_w2v.kv",
    BASE / "models" / "w2v_authors" / "dostoevsky_w2v.kv",
    BASE / "models" / "w2v_authors" / "leskov_w2v.kv",
    BASE / "models" / "w2v_authors" / "nekrasov_w2v.kv",
    BASE / "models" / "w2v_authors" / "saltykov-schedrin_w2v.kv",
    BASE / "models" / "w2v_authors" / "turgenev_w2v.kv",
    # НКРЯ (ruscorpora / gensim-data): добавляем в сравнение
    BASE / "models" / "w2v_authors" / "rucorpora.kv",
]

MODEL_NAMES = [p.stem for p in MODEL_PATHS]
MODEL_NAMES


In [ ]:
models: list[KeyedVectors] = [KeyedVectors.load_word2vec_format(str(p), binary=False) for p in MODEL_PATHS]

vocab_sets = [set(m.key_to_index.keys()) for m in models]
common_vocab = sorted(set.intersection(*vocab_sets))

pd.DataFrame(
    {
        "model": ["rucorpora" if n == "rucorpora" else n for n in MODEL_NAMES],
        "vocab_size": [len(m) for m in models],
        "common_vocab": [len(common_vocab)] * len(models),
    }
)


In [ ]:
# Матрицы векторов в одном и том же порядке слов

X = []
for m in models:
    mat = np.vstack([m[w] for w in common_vocab]).astype(np.float64)
    X.append(mat)

# L2-нормировка строк для cosine
Xn = [mat / np.linalg.norm(mat, axis=1, keepdims=True) for mat in X]

[len(common_vocab), Xn[0].shape]


## 1) Orthogonal Procrustes (выровнять пары моделей и сравнить слово с самим собой)

Шаги:
- для каждой пары моделей найти ортогональное преобразование
- применить к одной модели
- для каждого слова посчитать cosine similarity с самим собой во второй модели
- сохранить mean, median
- построить распределение расстояний \(1-\cos\)
- сохранить графики: матрица гистограмм по всем парам + матрица средних cosine distance


### Procrustes: слова, сильнее всего изменившиеся по контексту

После выравнивания (Procrustes) считаем две метрики:
- **Cosine displacement**: \(1 - \cos(\hat{x}_w, y_w)\). Больше → слово «сдвинулось» сильнее.
- **Local Neighborhood Distance (LND)**: сравниваем локальную геометрию вокруг слова через расстояния до объединения top-k соседей (k=15). Больше → сильнее изменилась локальная структура.


In [ ]:
from itertools import combinations

# Orthogonal Procrustes через SVD: хотим R = argmin ||A R - B||, R ортогональна

def procrustes_rotation(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    M = A.T @ B
    U, _, Vt = np.linalg.svd(M, full_matrices=False)
    return U @ Vt

pair_rows = []
pair_dists: dict[tuple[int, int], np.ndarray] = {}

for i, j in combinations(range(len(models)), 2):
    A = Xn[i]
    B = Xn[j]
    R = procrustes_rotation(A, B)
    A_aligned = A @ R

    cos_self = np.sum(A_aligned * B, axis=1)
    dist = 1.0 - cos_self

    pair_dists[(i, j)] = dist
    pair_rows.append(
        {
            "model_a": MODEL_NAMES[i],
            "model_b": MODEL_NAMES[j],
            "mean_cos": float(np.mean(cos_self)),
            "median_cos": float(np.median(cos_self)),
            "mean_cos_dist": float(np.mean(dist)),
            "median_cos_dist": float(np.median(dist)),
        }
    )

procrustes_df = pd.DataFrame(pair_rows).sort_values(["model_a", "model_b"])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = len(MODEL_NAMES)

# --- общая шкала X для всех графиков ---
all_vals = np.concatenate([pair_dists[k] for k in pair_dists.keys()])
x_min = max(0.0, np.min(all_vals))
x_max = min(2.0, np.max(all_vals))

pad = 0.03 * (x_max - x_min) if x_max > x_min else 0.02
x_min_plot = max(0.0, x_min - pad)
x_max_plot = min(2.0, x_max + pad)

# --- нижний треугольник ---
fig, axes = plt.subplots(
    n, n,
    figsize=(2.8 * n, 2.8 * n),
    sharex=True,
    sharey=True
)

if n == 1:
    axes = np.array([[axes]])

for i in range(n):
    for j in range(n):
        ax = axes[i, j]

        # верхний треугольник и диагональ скрываем
        if j >= i:
            ax.axis("off")
            continue

        dist = pair_dists[(j, i)] if (j, i) in pair_dists else pair_dists[(i, j)]

        ax.hist(
            dist,
            bins=25,
            density=False,
            alpha=1.0,
            edgecolor="black",
            linewidth=0.6
        )

        ax.set_title(
            f"{MODEL_NAMES[i]} × {MODEL_NAMES[j]}",
            fontsize=10,
            pad=6
        )

        ax.set_xlim(x_min_plot, x_max_plot)
        ax.grid(axis="y", alpha=0.25, linewidth=0.6)

        if i == n - 1:
            ax.set_xlabel("1 - cosine", fontsize=10)
        else:
            ax.set_xlabel("")

        if j == 0:
            ax.set_ylabel("Count", fontsize=10)
        else:
            ax.set_ylabel("")

fig.suptitle(
    "Pairwise distributions of Procrustes-aligned distances",
    fontsize=14,
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

fig_path = OUT_DIR / "procrustes_distance_triangle_density.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

fig_path

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

K_PRO = 15
TOP_N = 30


def topk_neighbor_indices(mat: np.ndarray, k: int, block: int = 256) -> list[np.ndarray]:
    n = mat.shape[0]
    res = []
    for start in range(0, n, block):
        end = min(n, start + block)
        sims = mat[start:end] @ mat.T
        sims[np.arange(end - start), np.arange(start, end)] = -np.inf

        idx = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
        vals = np.take_along_axis(sims, idx, axis=1)
        order = np.argsort(-vals, axis=1)
        idx = np.take_along_axis(idx, order, axis=1)

        res.extend([idx[r].astype(np.int32, copy=False) for r in range(idx.shape[0])])
    return res


def lnd_for_pair(
    A_aligned: np.ndarray,
    B: np.ndarray,
    neigh_A: list[np.ndarray],
    neigh_B: list[np.ndarray],
) -> np.ndarray:
    n = A_aligned.shape[0]
    out = np.empty(n, dtype=np.float64)
    for t in range(n):
        union_idx = np.unique(np.concatenate([neigh_A[t], neigh_B[t]]))
        da = 1.0 - (A_aligned[t] @ A_aligned[union_idx].T)
        db = 1.0 - (B[t] @ B[union_idx].T)
        out[t] = float(np.mean(np.abs(da - db)))
    return out


# -------------------------------------------------
# 1. Сначала считаем displacement и LND по всем парам
# -------------------------------------------------
pair_displacement = {}
pair_lnd = {}
pair_lnd_top_words = {}
pair_disp_top_words = {}

for i, j in combinations(range(len(models)), 2):
    A = Xn[i]
    B = Xn[j]

    R = procrustes_rotation(A, B)
    A_aligned = A @ R

    # cosine displacement: берём из уже посчитанного dist (1 - cosine) после Procrustes
    displacement = pair_dists[(i, j)]
    pair_displacement[(i, j)] = displacement

    # LND
    neigh_A = topk_neighbor_indices(A_aligned, K_PRO)
    neigh_B = topk_neighbor_indices(B, K_PRO)
    lnd = lnd_for_pair(A_aligned, B, neigh_A, neigh_B)
    pair_lnd[(i, j)] = lnd

    # сразу запомним top слова, чтобы потом не пересчитывать
    disp_order = np.argsort(-displacement)
    lnd_order = np.argsort(-lnd)

    pair_disp_top_words[(i, j)] = [common_vocab[t] for t in disp_order[:TOP_N]]
    pair_lnd_top_words[(i, j)] = [common_vocab[t] for t in lnd_order[:TOP_N]]


# -------------------------------------------------
# 2. СТРОИМ МАТРИЦУ РАСПРЕДЕЛЕНИЙ LND УГОЛКОМ
# -------------------------------------------------
n = len(MODEL_NAMES)

all_vals_lnd = np.concatenate([pair_lnd[k] for k in pair_lnd.keys()])
x_min = max(0.0, np.min(all_vals_lnd))
x_max = np.max(all_vals_lnd)

pad = 0.03 * (x_max - x_min) if x_max > x_min else 0.01
x_min_plot = max(0.0, x_min - pad)
x_max_plot = x_max + pad

bin_edges_lnd = np.linspace(x_min_plot, x_max_plot, 31)

fig, axes = plt.subplots(
    n, n,
    figsize=(2.8 * n, 2.8 * n),
    sharex=True,
    sharey=True
)

if n == 1:
    axes = np.array([[axes]])

for i in range(n):
    for j in range(n):
        ax = axes[i, j]

        # скрываем диагональ и верхний треугольник
        if j >= i:
            ax.axis("off")
            continue

        a, b = (j, i) if (j, i) in pair_lnd else (i, j)
        vals = pair_lnd[(a, b)]

        ax.hist(
            vals,
            bins=bin_edges_lnd,
            density=True,
            alpha=1.0,
            edgecolor="black",
            linewidth=0.6
        )

        ax.set_title(
            f"{MODEL_NAMES[i]} × {MODEL_NAMES[j]}",
            fontsize=10,
            pad=6
        )

        ax.set_xlim(x_min_plot, x_max_plot)
        ax.grid(axis="y", alpha=0.25, linewidth=0.6)

        if i == n - 1:
            ax.set_xlabel("LND", fontsize=10)
        else:
            ax.set_xlabel("")

        if j == 0:
            ax.set_ylabel("Density", fontsize=10)
        else:
            ax.set_ylabel("")

fig.suptitle(
    f"Pairwise distributions of LND after Procrustes alignment (k={K_PRO})",
    fontsize=14,
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

fig_path = OUT_DIR / f"procrustes_lnd_triangle_density_k{K_PRO}.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

fig_path

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

n = len(MODEL_NAMES)

# -----------------------------------
# 1. Собираем попарные матрицы
# -----------------------------------
mean_disp_mat = np.full((n, n), np.nan)
median_disp_mat = np.full((n, n), np.nan)
mean_lnd_mat = np.full((n, n), np.nan)
median_lnd_mat = np.full((n, n), np.nan)

for (i, j), vals in pair_displacement.items():
    mean_disp_mat[i, j] = mean_disp_mat[j, i] = np.mean(vals)
    median_disp_mat[i, j] = median_disp_mat[j, i] = np.median(vals)

for (i, j), vals in pair_lnd.items():
    mean_lnd_mat[i, j] = mean_lnd_mat[j, i] = np.mean(vals)
    median_lnd_mat[i, j] = median_lnd_mat[j, i] = np.median(vals)

# диагональ можно обнулить или оставить nan
np.fill_diagonal(mean_disp_mat, 0.0)
np.fill_diagonal(median_disp_mat, 0.0)
np.fill_diagonal(mean_lnd_mat, 0.0)
np.fill_diagonal(median_lnd_mat, 0.0)

# сохранить таблицы
pd.DataFrame(mean_disp_mat, index=MODEL_NAMES, columns=MODEL_NAMES).to_csv(
    OUT_DIR / "heatmap_mean_cosine_displacement.csv"
)
pd.DataFrame(median_disp_mat, index=MODEL_NAMES, columns=MODEL_NAMES).to_csv(
    OUT_DIR / "heatmap_median_cosine_displacement.csv"
)
pd.DataFrame(mean_lnd_mat, index=MODEL_NAMES, columns=MODEL_NAMES).to_csv(
    OUT_DIR / f"heatmap_mean_lnd_k{K_PRO}.csv"
)
pd.DataFrame(median_lnd_mat, index=MODEL_NAMES, columns=MODEL_NAMES).to_csv(
    OUT_DIR / f"heatmap_median_lnd_k{K_PRO}.csv"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

short_labels = [
    name.replace("_w2v", "").replace("saltykov-schedrin", "s.-schedrin")
    for name in MODEL_NAMES
]

def draw_heatmap(ax, mat, labels, title, cmap="viridis", fmt=".3f"):
    plot_mat = mat.copy().astype(float)
    n = len(labels)

    im = ax.imshow(
        plot_mat,
        cmap=cmap,
        aspect="equal",
        interpolation="none",   # вместо "nearest"
        resample=False
    )

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)

    for i in range(n):
        for j in range(n):
            val = plot_mat[i, j]
            if np.isnan(val):
                continue
            ax.text(
                j, i, f"{val:{fmt}}",
                ha="center", va="center",
                fontsize=8, color="black"
            )

    # жёстко отключаем любую сетку
    ax.grid(False)
    ax.grid(which="minor", visible=False)
    ax.grid(which="major", visible=False)
    ax.minorticks_off()

    # убрать рамку осей, иногда тоже визуально мешает
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(title, fontsize=11, pad=8)
    return im


fig, axes = plt.subplots(2, 2, figsize=(10, 8))

im1 = draw_heatmap(axes[0, 0], mean_disp_mat, short_labels, "Mean cosine displacement")
im2 = draw_heatmap(axes[0, 1], median_disp_mat, short_labels, "Median cosine displacement")
im3 = draw_heatmap(axes[1, 0], mean_lnd_mat, short_labels, f"Mean LND (k={K_PRO})")
im4 = draw_heatmap(axes[1, 1], median_lnd_mat, short_labels, f"Median LND (k={K_PRO})")

# colorbar
for ax, im in zip(axes.flat, [im1, im2, im3, im4]):
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("Pairwise comparison of aligned embedding spaces", fontsize=13)

plt.tight_layout()

fig_path = OUT_DIR / "pairwise_heatmaps_clean_classic.png"
plt.savefig(fig_path, dpi=300)
plt.show()

fig_path

In [ ]:
# -------------------------------------------------
# 3. ПОСЛЕ ГРАФИКА СОБИРАЕМ DF И СОХРАНЯЕМ TOP СЛОВА
# -------------------------------------------------
procrustes_word_rows = []

for i, j in combinations(range(len(models)), 2):
    displacement = pair_displacement[(i, j)]
    lnd = pair_lnd[(i, j)]

    disp_df = pd.DataFrame(
        {
            "word": common_vocab,
            "model_a": MODEL_NAMES[i],
            "model_b": MODEL_NAMES[j],
            "cosine_displacement": displacement,
        }
    ).sort_values("cosine_displacement", ascending=False)

    disp_df.to_csv(
        OUT_DIR / f"procrustes_cosine_displacement__{MODEL_NAMES[i]}__{MODEL_NAMES[j]}.csv",
        index=False
    )
    disp_df.head(TOP_N).to_csv(
        OUT_DIR / f"procrustes_cosine_displacement_top{TOP_N}__{MODEL_NAMES[i]}__{MODEL_NAMES[j]}.csv",
        index=False
    )

    lnd_df = pd.DataFrame(
        {
            "word": common_vocab,
            "model_a": MODEL_NAMES[i],
            "model_b": MODEL_NAMES[j],
            "lnd": lnd,
        }
    ).sort_values("lnd", ascending=False)

    lnd_df.to_csv(
        OUT_DIR / f"procrustes_lnd_k{K_PRO}__{MODEL_NAMES[i]}__{MODEL_NAMES[j]}.csv",
        index=False
    )
    lnd_df.head(TOP_N).to_csv(
        OUT_DIR / f"procrustes_lnd_top{TOP_N}_k{K_PRO}__{MODEL_NAMES[i]}__{MODEL_NAMES[j]}.csv",
        index=False
    )

    procrustes_word_rows.append(
        {
            "model_a": MODEL_NAMES[i],
            "model_b": MODEL_NAMES[j],
            "mean_cosine_displacement": float(np.mean(displacement)),
            "median_cosine_displacement": float(np.median(displacement)),
            "mean_lnd": float(np.mean(lnd)),
            "median_lnd": float(np.median(lnd)),
            "top_cosine_displacement": ", ".join(pair_disp_top_words[(i, j)][:10]),
            "top_lnd": ", ".join(pair_lnd_top_words[(i, j)][:10]),
        }
    )

summary_df = pd.DataFrame(procrustes_word_rows)
summary_df.to_csv(OUT_DIR / "procrustes_top_changed_words_summary.csv", index=False)
summary_df

## 2) Gromov–Wasserstein distance (POT / ot)

Шаги:
- использовать `POT` (`ot`)
- для каждой пары моделей считать расстояние между матрицами cosine distance
- построить итоговую попарную матрицу расстояний между моделями

Замечание: GW требует \(N\times N\) cost-матрицы; для больших пересечений это очень тяжело по памяти/времени, поэтому здесь фиксируется число слов для GW.


In [ ]:
len(common_vocab)

In [ ]:
import ot

GW_N_WORDS = 2000  # можно увеличить, но память растёт как O(N^2)

gw_vocab = common_vocab[: min(len(common_vocab), GW_N_WORDS)]
Xg = []
for m in models:
    mat = np.vstack([m[w] for w in gw_vocab]).astype(np.float64)
    mat = mat / np.linalg.norm(mat, axis=1, keepdims=True)
    Xg.append(mat)

def cosine_cost(M: np.ndarray) -> np.ndarray:
    C = 1.0 - (M @ M.T)
    C[C < 0] = 0.0
    return C

Cg = [cosine_cost(M) for M in Xg]

p = np.ones(len(gw_vocab), dtype=np.float64) / len(gw_vocab)

n = len(models)
GW = np.zeros((n, n), dtype=np.float64)

for i in range(n):
    for j in range(i + 1, n):
        gw2 = ot.gromov.gromov_wasserstein2(Cg[i], Cg[j], p, p, loss_fun="square_loss")
        GW[i, j] = float(gw2)
        GW[j, i] = float(gw2)

GW_df = pd.DataFrame(GW, index=MODEL_NAMES, columns=MODEL_NAMES)
GW_df.to_csv(OUT_DIR / "gw_distance_matrix.csv")

plt.figure(figsize=(1.1 * n + 4, 1.0 * n + 3))
sns.heatmap(GW_df, annot=True, fmt=".4f", cmap="magma")
plt.title(f"Gromov–Wasserstein distance (N={len(gw_vocab)})")
plt.tight_layout()
fig_path = OUT_DIR / "gw_distance_matrix.png"
plt.savefig(fig_path, dpi=200)
plt.show()

GW_df


## 3) Локальная структура через соседей (Jaccard overlap)

Шаги:
- для каждой лексемы взять top-k ближайших соседей (фиксируем k=15)
- сравнить множества соседей между моделями через Jaccard overlap
- посчитать распределение overlap по словам
- построить и сохранить графики распределения


In [ ]:
K = 15

neighbors: list[dict[str, set[str]]] = []
for m in models:
    d: dict[str, set[str]] = {}
    for w in common_vocab:
        sims = m.most_similar(w, topn=K + 1)
        nn = [t for t, _ in sims if t != w][:K]
        d[w] = set(nn)
    neighbors.append(d)

len(neighbors), len(neighbors[0])


In [ ]:
from itertools import combinations

jac_rows = []
jac_dists: dict[tuple[int, int], np.ndarray] = {}

for i, j in combinations(range(len(models)), 2):
    overlaps = []
    for w in common_vocab:
        A = neighbors[i][w]
        B = neighbors[j][w]
        overlaps.append(len(A & B) / len(A | B))

    overlaps = np.asarray(overlaps, dtype=np.float64)
    jac_dists[(i, j)] = overlaps

    jac_rows.append(
        {
            "model_a": MODEL_NAMES[i],
            "model_b": MODEL_NAMES[j],
            "mean_jaccard": float(np.mean(overlaps)),
            "median_jaccard": float(np.median(overlaps)),
        }
    )

jaccard_df = pd.DataFrame(jac_rows).sort_values(["model_a", "model_b"])
jaccard_df.to_csv(OUT_DIR / "neighbors_jaccard_pair_stats.csv", index=False)
jaccard_df


### Neighborhood comparison: ранжирование слов (Jaccard + RBO)

- **Jaccard overlap**: пересечение множеств соседей. Меньше → локальный контекст сильнее различается.
- **RBO (Rank-Biased Overlap)**: пересечение *ранжированных* списков соседей с учётом порядка. Меньше → сильнее различаются ближайшие соседи (особенно верх списка).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = len(MODEL_NAMES)

x_min_plot = 0.0
x_max_plot = 1.0

# Общие границы бинов для всех графиков
bin_edges = np.linspace(0.0, 1.0, 31)   # 30 одинаковых бинов

fig, axes = plt.subplots(
    n, n,
    figsize=(2.8 * n, 2.8 * n),
    sharex=True,
    sharey=True
)

if n == 1:
    axes = np.array([[axes]])

for i in range(n):
    for j in range(n):
        ax = axes[i, j]

        # Верхний треугольник и диагональ скрываем
        if j >= i:
            ax.axis("off")
            continue

        a, b = (j, i) if (j, i) in jac_dists else (i, j)
        overlaps = jac_dists[(a, b)]

        ax.hist(
            overlaps,
            bins=bin_edges,
            density=True,
            alpha=1.0,
            edgecolor="black",
            linewidth=0.6
        )

        ax.set_title(
            f"{MODEL_NAMES[i]} × {MODEL_NAMES[j]}",
            fontsize=10,
            pad=6
        )

        ax.set_xlim(x_min_plot, x_max_plot)
        ax.grid(axis="y", alpha=0.25, linewidth=0.6)

        if i == n - 1:
            ax.set_xlabel("Jaccard overlap", fontsize=10)
        else:
            ax.set_xlabel("")

        if j == 0:
            ax.set_ylabel("Density", fontsize=10)
        else:
            ax.set_ylabel("")

fig.suptitle(
    "Pairwise distributions of neighborhood overlap",
    fontsize=14,
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

fig_path = OUT_DIR / "neighbors_jaccard_triangle_density.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

fig_path

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

# Пер-слово: Jaccard overlap и RBO по парам моделей

RBO_P = 0.9
TOP_N = 50


def rbo_ext(list_a: list[str], list_b: list[str], p: float = 0.9) -> float:
    """
    Extended RBO for finite ranked lists.
    Больше -> списки ближе; меньше -> сильнее различаются верхние соседи.
    Предполагается, что списки конечны; работает и при одинаковой длине top-k.
    """
    if not list_a or not list_b:
        return 0.0

    k = min(len(list_a), len(list_b))

    seen_a = set()
    seen_b = set()
    overlaps = np.zeros(k, dtype=np.float64)

    for d in range(1, k + 1):
        seen_a.add(list_a[d - 1])
        seen_b.add(list_b[d - 1])
        overlaps[d - 1] = len(seen_a & seen_b) / d

    # Основная часть + хвост на глубине k
    rbo_min = (1.0 - p) * np.sum((p ** np.arange(k)) * overlaps)
    rbo_tail = (p ** k) * overlaps[-1]

    return float(rbo_min + rbo_tail)


neighbors_ranked: list[dict[str, list[str]]] = []
for m in models:
    d_rank: dict[str, list[str]] = {}
    for w in common_vocab:
        sims = m.most_similar(w, topn=K + 1)
        nn = [t for t, _ in sims if t != w][:K]
        d_rank[w] = nn
    neighbors_ranked.append(d_rank)

word_metrics_rows = []

# Для дальнейших triangular plots / heatmaps
pair_jaccard = {}
pair_rbo = {}

for i, j in combinations(range(len(models)), 2):
    jac = np.asarray(
        [
            len(neighbors[i][w] & neighbors[j][w]) / len(neighbors[i][w] | neighbors[j][w])
            for w in common_vocab
        ],
        dtype=np.float64,
    )

    rbo_vals = np.asarray(
        [rbo_ext(neighbors_ranked[i][w], neighbors_ranked[j][w], p=RBO_P) for w in common_vocab],
        dtype=np.float64,
    )

    pair_jaccard[(i, j)] = jac
    pair_rbo[(i, j)] = rbo_vals

    df = pd.DataFrame(
        {
            "word": common_vocab,
            "model_a": MODEL_NAMES[i],
            "model_b": MODEL_NAMES[j],
            "jaccard_overlap": jac,
            "rbo": rbo_vals,
        }
    )

    df.to_csv(
        OUT_DIR / f"neighbors_word_metrics__{MODEL_NAMES[i]}__{MODEL_NAMES[j]}.csv",
        index=False
    )

    df.sort_values("jaccard_overlap", ascending=True).head(TOP_N).to_csv(
        OUT_DIR / f"neighbors_min_jaccard_top{TOP_N}__{MODEL_NAMES[i]}__{MODEL_NAMES[j]}.csv",
        index=False,
    )

    df.sort_values("rbo", ascending=True).head(TOP_N).to_csv(
        OUT_DIR / f"neighbors_min_rbo_top{TOP_N}__{MODEL_NAMES[i]}__{MODEL_NAMES[j]}.csv",
        index=False,
    )

    word_metrics_rows.append(df)

neighbors_word_metrics_long = pd.concat(word_metrics_rows, ignore_index=True)
neighbors_word_metrics_long.to_csv(OUT_DIR / "neighbors_word_metrics_long.csv", index=False)

neighbors_word_metrics_long.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

n = len(MODEL_NAMES)

# --- собираем только mean матрицы ---
mean_jac_mat = np.full((n, n), np.nan)
mean_rbo_mat = np.full((n, n), np.nan)

for (i, j), vals in pair_jaccard.items():
    mean_jac_mat[i, j] = mean_jac_mat[j, i] = np.mean(vals)

for (i, j), vals in pair_rbo.items():
    mean_rbo_mat[i, j] = mean_rbo_mat[j, i] = np.mean(vals)

# диагональ исключаем
np.fill_diagonal(mean_jac_mat, np.nan)
np.fill_diagonal(mean_rbo_mat, np.nan)

# сохранить таблицы
pd.DataFrame(mean_jac_mat, index=MODEL_NAMES, columns=MODEL_NAMES).to_csv(
    OUT_DIR / f"neighbors_mean_jaccard_matrix_k{K}_no_diag.csv"
)
pd.DataFrame(mean_rbo_mat, index=MODEL_NAMES, columns=MODEL_NAMES).to_csv(
    OUT_DIR / f"neighbors_mean_rbo_matrix_k{K}_p{RBO_P}_no_diag.csv"
)

# короткие подписи
short_labels = [
    name.replace("_w2v", "").replace("saltykov-schedrin", "s.-schedrin")
    for name in MODEL_NAMES
]

def draw_heatmap(ax, mat, labels, title, cmap="viridis", fmt=".3f"):
    plot_mat = mat.copy().astype(float)

    im = ax.imshow(
        plot_mat,
        cmap=cmap,
        aspect="equal",
        interpolation="none"
    )

    n_local = len(labels)

    ax.set_xticks(np.arange(n_local))
    ax.set_yticks(np.arange(n_local))
    ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)

    # значения (без диагонали)
    for i in range(n_local):
        for j in range(n_local):
            val = plot_mat[i, j]
            if np.isnan(val):
                continue
            ax.text(
                j, i, format(val, fmt),
                ha="center", va="center",
                fontsize=8,
                color="black"
            )

    ax.set_title(title, fontsize=11, pad=8)

    return im


# --- 2 heatmap ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))


im1 = draw_heatmap(
    axes[0],
    mean_jac_mat,
    short_labels,
    f"Mean Jaccard (k={K})"
)

im2 = draw_heatmap(
    axes[1],
    mean_rbo_mat,
    short_labels,
    f"Mean RBO (k={K}, p={RBO_P})"
)

for ax, im in zip(axes, [im1, im2]):
    # жёстко отключаем любую сетку
    ax.grid(False)
    ax.grid(which="minor", visible=False)
    ax.grid(which="major", visible=False)
    ax.minorticks_off()
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("Pairwise neighborhood similarity (mean, no diagonal)", fontsize=13)

plt.tight_layout()

fig_path = OUT_DIR / f"neighbors_pairwise_heatmaps_mean_k{K}_p{RBO_P}_no_diag.png"
plt.savefig(fig_path, dpi=300)
plt.show()

fig_path

## Просмотр «контекстов» слова в двух моделях

Выберите две модели и слово, чтобы увидеть:
- top-k соседей (ранжированный список)
- пересечение соседей
- метрики различия (если они считались выше): displacement, LND, Jaccard, RBO


In [ ]:
# Выберите пару моделей и слово из common_vocab

model_a = MODEL_NAMES[0]
model_b = MODEL_NAMES[1]
word = common_vocab[0]

k = 15

ia = MODEL_NAMES.index(model_a)
ib = MODEL_NAMES.index(model_b)

w_idx = common_vocab.index(word)

# Соседи (ранжированные) + пересечение
na = [t for t, _ in models[ia].most_similar(word, topn=k + 1) if t != word][:k]
nb = [t for t, _ in models[ib].most_similar(word, topn=k + 1) if t != word][:k]

print("model_a:", model_a)
print("model_b:", model_b)
print("word:", word)
print()

print(f"Top-{k} neighbors in A:")
print(na)
print()

print(f"Top-{k} neighbors in B:")
print(nb)
print()

inter = sorted(set(na) & set(nb))
print(f"Intersection size: {len(inter)}")
print(inter)
print()

# Метрики для этого слова (если соответствующие словари/переменные уже посчитаны выше)
key = (min(ia, ib), max(ia, ib))

if "pair_dists" in globals():
    disp = pair_dists[key][w_idx]
    print("Procrustes cosine displacement (1 - cosine):", float(disp))

if "pair_lnd" in globals() and key in pair_lnd:
    print("Procrustes LND:", float(pair_lnd[key][w_idx]))

# Jaccard по множествам соседей (как в блоке overlap)
sa = set(na)
sb = set(nb)
print("Jaccard overlap:", len(sa & sb) / len(sa | sb))

# RBO (если функция определена выше)
if "rbo" in globals():
    print("RBO:", float(rbo(na, nb, p=RBO_P)))


In [ ]:
# Топ-10 наиболее и наименее «подвижных» слов среди всех авторов (агрегация по всем парам моделей)

metric = "jaccard"  # displacement | lnd | jaccard | rbo
TOP_N = 10

# Считаем подвижность ТОЛЬКО относительно НКРЯ-модели (rucorpora)
ref_name = "rucorpora"
ref_idx = MODEL_NAMES.index(ref_name)

pair_values: dict[str, list[np.ndarray]] = {
    "displacement": [pair_dists[k] for k in pair_dists.keys() if ref_idx in k],
    "lnd": [pair_lnd[k] for k in pair_lnd.keys() if ref_idx in k] if "pair_lnd" in globals() else [],
}

if metric in pair_values:
    # Чем больше distance, тем выше «подвижность»
    vals = np.vstack(pair_values[metric])  # (n_ref_pairs, n_words)
    mobility = np.mean(vals, axis=0)
    scores = pd.DataFrame({"word": common_vocab, "mobility": mobility})

else:
    # Для overlap-метрик «подвижность» = 1 - mean(overlap), но только для пар (author × rucorpora)
    dfm = neighbors_word_metrics_long.copy()  # columns: word, model_a, model_b, jaccard_overlap, rbo
    dfm = dfm[(dfm["model_a"] == ref_name) | (dfm["model_b"] == ref_name)]

    col = "jaccard_overlap" if metric == "jaccard" else "rbo"
    scores = (
        dfm.groupby("word")[col]
        .mean()
        .rename("mean_overlap")
        .reset_index()
    )

    scores["mobility"] = 1.0 - scores["mean_overlap"]
    scores = scores[["word", "mobility"]]

scores = scores.sort_values("mobility", ascending=False).reset_index(drop=True)
most = scores.head(TOP_N).copy()
least = scores.tail(TOP_N).sort_values("mobility", ascending=True).reset_index(drop=True).copy()

# Компактная таблица (удобно вставлять в Word)
out_table = pd.DataFrame(
    {
        "least_mobile_word": least["word"],
        "least_mobile_score": least["mobility"].round(4),
        "most_mobile_word": most["word"],
        "most_mobile_score": most["mobility"].round(4),
    }
)

xlsx_path = OUT_DIR / f"mobility_top10_{metric}.xlsx"
out_table.to_excel(xlsx_path, sheet_name="top10", index=False)

xlsx_path, out_table

In [ ]:
# Все авторы vs НКРЯ (rucorpora): топ-10 самых/наименее подвижных слов по всем метрикам
# (создаёт отдельный .xlsx на каждого автора; листы = метрики)

REF_NAME = "rucorpora"
TOP_N = 10

K_PRO_LOCAL = 15   # k для LND
K_NEIGH = 50       # k соседей для Jaccard/RBO
RBO_P = 0.9

ir = MODEL_NAMES.index(REF_NAME)
vocab_ref = set(models[ir].key_to_index.keys())

author_names = [n for n in MODEL_NAMES if n != REF_NAME]
out_paths = []

for AUTHOR_NAME in author_names:
    ia = MODEL_NAMES.index(AUTHOR_NAME)
    vocab_author = set(models[ia].key_to_index.keys())
    vocab_pair = sorted(vocab_author & vocab_ref)

    print("\nAUTHOR:", AUTHOR_NAME, "|V|=", len(vocab_author), "PAIR=", len(vocab_pair))

    # Матрицы в одном порядке слов (только пересечение)
    A = np.vstack([models[ia][w] for w in vocab_pair]).astype(np.float64)
    B = np.vstack([models[ir][w] for w in vocab_pair]).astype(np.float64)
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)

    # Procrustes displacement
    R = procrustes_rotation(A, B)
    A_aligned = A @ R
    cos_self = np.sum(A_aligned * B, axis=1)
    displacement = 1.0 - cos_self

    # LND
    neigh_A = topk_neighbor_indices(A_aligned, K_PRO_LOCAL)
    neigh_B = topk_neighbor_indices(B, K_PRO_LOCAL)
    lnd = lnd_for_pair(A_aligned, B, neigh_A, neigh_B)

    # Overlap-метрики (Jaccard, RBO) по соседям в исходных моделях
    jacc = np.empty(len(vocab_pair), dtype=np.float64)
    rbo_vals = np.empty(len(vocab_pair), dtype=np.float64)

    for t, w in enumerate(vocab_pair):
        na = [x for x, _ in models[ia].most_similar(w, topn=K_NEIGH + 1) if x != w][:K_NEIGH]
        nb = [x for x, _ in models[ir].most_similar(w, topn=K_NEIGH + 1) if x != w][:K_NEIGH]

        sa = set(na)
        sb = set(nb)
        jacc[t] = len(sa & sb) / len(sa | sb) if (sa or sb) else 0.0
        rbo_vals[t] = rbo_ext(na, nb, p=RBO_P)

    df_scores = pd.DataFrame(
        {
            "word": vocab_pair,
            "displacement": displacement,
            "lnd": lnd,
            "jaccard": 1.0 - jacc,
            "rbo": 1.0 - rbo_vals,
        }
    )

    xlsx_path = OUT_DIR / f"mobility_author_vs_rucorpora__{AUTHOR_NAME}.xlsx"

    with pd.ExcelWriter(xlsx_path) as writer:
        for metric in ["displacement", "lnd", "jaccard", "rbo"]:
            scores = (
                df_scores[["word", metric]]
                .rename(columns={metric: "mobility"})
                .sort_values("mobility", ascending=False)
            )
            most = scores.head(TOP_N).copy()
            least = scores.tail(TOP_N).sort_values("mobility", ascending=True).reset_index(drop=True).copy()

            out_table = pd.DataFrame(
                {
                    "least_mobile_word": least["word"],
                    "least_mobile_score": least["mobility"].round(4),
                    "most_mobile_word": most["word"].values,
                    "most_mobile_score": most["mobility"].round(4).values,
                }
            )
            out_table.to_excel(writer, sheet_name=metric, index=False)

    out_paths.append(xlsx_path)

out_paths

## PaCMAP: «одно слово» в разных пространствах (с Procrustes-доворотом)

Идея: берём вектор выбранного слова в каждой модели, доворотом (Procrustes) выравниваем все пространства к одной опорной модели и визуализируем получившиеся точки в 2D. Близко = слово используется похоже, далеко = контекст отличается.


In [ ]:
import pacmap

# Выберите слово и опорную модель, к которой выравниваем остальные пространства
word = 'душа_NOUN'
ref_model = 'rucorpora'

# Сколько соседей показывать для каждого автора
k = 3

ref_idx = MODEL_NAMES.index(ref_model)
w_idx = common_vocab.index(word)

# Собираем: слово + его top-k соседей в каждой модели
rows = []

for m_idx, name in enumerate(MODEL_NAMES):
    # Procrustes-доворот к ref (один раз на модель)
    if m_idx == ref_idx:
        R = None
    else:
        R = procrustes_rotation(Xn[m_idx], Xn[ref_idx])

    # Вектор самого слова (берём из текущей модели; нормируем; затем доворот)
    v_word = models[m_idx][word].astype(np.float64)
    v_word = v_word / np.linalg.norm(v_word)
    v_word = v_word if R is None else (v_word @ R)
    rows.append({"model": name, "token": word, "kind": "word", "vec": v_word})

    # Соседи (ранжированный список). Важно: соседи могут быть вне common_vocab.
    nn = [t for t, _ in models[m_idx].most_similar(word, topn=k + 1) if t != word][:k]
    for t in nn:
        v = models[m_idx][t].astype(np.float64)
        v = v / np.linalg.norm(v)
        v = v if R is None else (v @ R)
        rows.append({"model": name, "token": t, "kind": "neighbor", "vec": v})

V = np.vstack([r["vec"] for r in rows]).astype(np.float64)

# PaCMAP на всех точках разом
n = V.shape[0]
neigh = max(2, min(15, n - 1))

emb = pacmap.PaCMAP(n_components=2, n_neighbors=neigh, MN_ratio=0.5, FP_ratio=2.0, random_state=42)
Y = emb.fit_transform(V)

plot_df = pd.DataFrame(
    {
        "x": Y[:, 0],
        "y": Y[:, 1],
        "model": [r["model"] for r in rows],
        "token": [r["token"] for r in rows],
        "kind": [r["kind"] for r in rows],
    }
)

plt.figure(figsize=(9, 7), dpi=300)

# Строгий базовый стиль: без контуров у маркеров
palette = dict(zip(MODEL_NAMES, sns.color_palette("tab10", n_colors=len(MODEL_NAMES))))

neighbors_df = plot_df[plot_df["kind"] == "neighbor"].copy()
words_df = plot_df[plot_df["kind"] == "word"].copy()

# Соседи: точки
sns.scatterplot(
    data=neighbors_df,
    x="x",
    y="y",
    hue="model",
    palette=palette,
    marker="o",
    s=55,
    alpha=1,
    linewidth=0,
)

# Ключевое слово: крест
sns.scatterplot(
    data=words_df,
    x="x",
    y="y",
    hue="model",
    palette=palette,
    marker="X",
    s=130,
    alpha=1,
    linewidth=0,
    legend=False,
)

# Подписи:
# - ключевое слово подписываем самим словом
# - соседей подписываем токенами
from adjustText import adjust_text

texts = []
for _, r in words_df.iterrows():
    texts.append(plt.text(r["x"], r["y"], word, fontsize=10, weight="bold"))

for _, r in neighbors_df.iterrows():
    texts.append(plt.text(r["x"], r["y"], r["token"], fontsize=8))

# Автоматически раздвигаем подписи, чтобы меньше перекрывались
adjust_text(texts, ax=plt.gca(), only_move={"text": "xy"})

plt.title(f"PaCMAP: '{word}' + top-{k} neighbors (выровнено к {ref_model})")
plt.xlabel("dim 1")
plt.ylabel("dim 2")

# Легенда: цвета = авторы
plt.legend(title="author", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()

fig_path = OUT_DIR / f"pacmap_word_neighbors__{word}__k{k}__ref_{ref_model}.png"
plt.savefig(fig_path, dpi=300)
plt.show()


## Приложение и тест значимости

In [ ]:
# Реплики моделей (seeds) — сравнение авторов по 4 метрикам (как в heatmap выше)
# Папка: models/w2v_authors/replicates

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gensim.models import KeyedVectors

# Статистика: SciPy (если есть), иначе пермутационный тест
try:
    from scipy.stats import mannwhitneyu
except Exception:
    mannwhitneyu = None

REPL_DIR = BASE / "models" / "w2v_authors" / "replicates"
rep_kv_paths = sorted(REPL_DIR.glob("*.kv"))
assert rep_kv_paths, f"Не найдено реплик .kv в {REPL_DIR}"

# --- загрузка реплик ---
rep_models: dict[str, list[KeyedVectors]] = {}
for p in rep_kv_paths:
    stem = p.stem
    author = stem.split("_seed", 1)[0]
    rep_models.setdefault(author, []).append(KeyedVectors.load_word2vec_format(str(p), binary=False))

AUTHORS = sorted(a for a in rep_models.keys() if a not in {"pushkin", "gogol"})
print("Authors:", AUTHORS)
print("Replicates per author:", {a: len(rep_models[a]) for a in AUTHORS})

# --- общий словарь для всех реплик (только выбранные AUTHORS) ---
vocab_sets = [set(m.key_to_index.keys()) for a in AUTHORS for m in rep_models[a]]
rep_common_vocab = sorted(set.intersection(*vocab_sets))

# Чтобы расчёты были выполнимы быстро: берём подвыборку общего словаря (фиксированный seed)
MAX_VOCAB = 5000
if len(rep_common_vocab) > MAX_VOCAB:
    rng = np.random.default_rng(42)
    rep_common_vocab = list(rng.choice(rep_common_vocab, size=MAX_VOCAB, replace=False))
    rep_common_vocab.sort()

print("Common vocab used:", len(rep_common_vocab))

# --- матрицы векторов (L2-норм.) для каждой реплики ---
rep_Xn: dict[tuple[str, int], np.ndarray] = {}
for a in AUTHORS:
    for r_idx, m in enumerate(rep_models[a]):
        mat = np.vstack([m[w] for w in rep_common_vocab]).astype(np.float64)
        mat = mat / np.linalg.norm(mat, axis=1, keepdims=True)
        rep_Xn[(a, r_idx)] = mat


def cliff_delta(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x)
    y = np.asarray(y)
    return float((x[:, None] > y[None, :]).mean() - (x[:, None] < y[None, :]).mean())


def _permutation_pvalue(x: np.ndarray, y: np.ndarray, n_perm: int = 20000, seed: int = 42) -> float:
    rng = np.random.default_rng(seed)
    x = np.asarray(x)
    y = np.asarray(y)
    obs = float(np.median(x) - np.median(y))
    z = np.concatenate([x, y])
    n_x = len(x)
    cnt = 0
    for _ in range(n_perm):
        rng.shuffle(z)
        d = float(np.median(z[:n_x]) - np.median(z[n_x:]))
        if abs(d) >= abs(obs):
            cnt += 1
    return (cnt + 1) / (n_perm + 1)


def compute_pair_metrics(A: np.ndarray, B: np.ndarray) -> dict[str, float]:
    R = procrustes_rotation(A, B)
    A_aligned = A @ R

    disp = 1.0 - np.sum(A_aligned * B, axis=1)

    neigh_A = topk_neighbor_indices(A_aligned, K_PRO)
    neigh_B = topk_neighbor_indices(B, K_PRO)
    lnd = lnd_for_pair(A_aligned, B, neigh_A, neigh_B)

    return {
        "mean_disp": float(np.mean(disp)),
        "median_disp": float(np.median(disp)),
        "mean_lnd": float(np.mean(lnd)),
        "median_lnd": float(np.median(lnd)),
    }


# --- распределения: внутри автора vs между авторами ---
within = {k: [] for k in ["mean_disp", "median_disp", "mean_lnd", "median_lnd"]}
between = {k: [] for k in ["mean_disp", "median_disp", "mean_lnd", "median_lnd"]}

# --- распределения: внутри автора vs между авторами (для статтеста) ---
# (оставляем как было)

# внутри автора
for a in AUTHORS:
    rN = len(rep_models[a])
    for i in range(rN):
        for j in range(i + 1, rN):
            met = compute_pair_metrics(rep_Xn[(a, i)], rep_Xn[(a, j)])
            for k, v in met.items():
                within[k].append(v)

# между авторами (для статтеста)
for ai, a in enumerate(AUTHORS):
    for bi, b in enumerate(AUTHORS):
        if bi < ai:
            continue
        if a == b:
            continue

        for i in range(len(rep_models[a])):
            for j in range(len(rep_models[b])):
                met = compute_pair_metrics(rep_Xn[(a, i)], rep_Xn[(b, j)])
                for k, v in met.items():
                    between[k].append(v)

# --- heatmap матрицы: все реплики всех авторов (реплики автора идут подряд) ---
rep_order: list[tuple[str, int]] = []
author_blocks: list[tuple[str, int, int]] = []  # (author, start, end)

cur = 0
for a in AUTHORS:
    start = cur
    for r_idx in range(len(rep_models[a])):
        rep_order.append((a, r_idx))
        cur += 1
    author_blocks.append((a, start, cur))

N = len(rep_order)

mean_disp_mat = np.full((N, N), np.nan)
median_disp_mat = np.full((N, N), np.nan)
mean_lnd_mat = np.full((N, N), np.nan)
median_lnd_mat = np.full((N, N), np.nan)

# считаем попарно метрики между всеми репликами
for i in range(N):
    A = rep_Xn[rep_order[i]]
    for j in range(i + 1, N):
        B = rep_Xn[rep_order[j]]
        met = compute_pair_metrics(A, B)

        mean_disp_mat[i, j] = mean_disp_mat[j, i] = met["mean_disp"]
        median_disp_mat[i, j] = median_disp_mat[j, i] = met["median_disp"]
        mean_lnd_mat[i, j] = mean_lnd_mat[j, i] = met["mean_lnd"]
        median_lnd_mat[i, j] = median_lnd_mat[j, i] = met["median_lnd"]

# диагональ — 0.0 как в основном heatmap-блоке
np.fill_diagonal(mean_disp_mat, 0.0)
np.fill_diagonal(median_disp_mat, 0.0)
np.fill_diagonal(mean_lnd_mat, 0.0)
np.fill_diagonal(median_lnd_mat, 0.0)

# подписи авторов по блокам (по 3 реплики подряд)
short_authors = [a.replace("saltykov-schedrin", "s.-schedrin") for a in AUTHORS]

block_centers = []
block_labels = []
block_edges = []
for a, start, end in author_blocks:
    block_centers.append((start + end - 1) / 2)
    block_labels.append(a.replace("saltykov-schedrin", "s.-schedrin"))
    block_edges.append(end - 0.5)


def draw_heatmap(ax, mat, title, cmap="viridis"):
    plot_mat = mat.copy().astype(float)

    im = ax.imshow(
        plot_mat,
        cmap=cmap,
        aspect="equal",
        interpolation="none",
        resample=False,
    )

    # подписи только по авторам (в центре блоков)
    ax.set_xticks(block_centers)
    ax.set_yticks(block_centers)
    ax.set_xticklabels(block_labels, rotation=35, ha="right", fontsize=9)
    ax.set_yticklabels(block_labels, fontsize=9)

    # границы между авторами
    for edge in block_edges[:-1]:
        ax.axvline(edge, color="black", linewidth=0.8, alpha=0.6)
        ax.axhline(edge, color="black", linewidth=0.8, alpha=0.6)

    ax.grid(False)
    ax.grid(which="minor", visible=False)
    ax.grid(which="major", visible=False)
    ax.minorticks_off()

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(title, fontsize=11, pad=8)
    return im


fig, axes = plt.subplots(2, 2, figsize=(10, 8))

im1 = draw_heatmap(axes[0, 0], mean_disp_mat, "Mean cosine displacement (replicates)")
im2 = draw_heatmap(axes[0, 1], median_disp_mat, "Median cosine displacement (replicates)")
im3 = draw_heatmap(axes[1, 0], mean_lnd_mat, f"Mean LND (k={K_PRO}, replicates)")
im4 = draw_heatmap(axes[1, 1], median_lnd_mat, f"Median LND (k={K_PRO}, replicates)")

for ax, im in zip(axes.flat, [im1, im2, im3, im4]):
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("Replicate-based author differences (replicates grouped by author)", fontsize=13)
plt.tight_layout()

fig_path = OUT_DIR / "replicates_pairwise_heatmaps_4metrics.png"
plt.savefig(fig_path, dpi=300)
plt.show()

# --- статистическая значимость: внутри автора vs между авторами ---
stat_rows = []
for k in ["mean_disp", "median_disp", "mean_lnd", "median_lnd"]:
    x = np.asarray(within[k], dtype=np.float64)
    y = np.asarray(between[k], dtype=np.float64)

    if mannwhitneyu is not None:
        p = float(mannwhitneyu(y, x, alternative="greater").pvalue)
        test = "Mann–Whitney U (between > within)"
    else:
        p = float(_permutation_pvalue(y, x))
        test = "Permutation test on median diff"

    stat_rows.append(
        {
            "metric": k,
            "within_n": int(len(x)),
            "between_n": int(len(y)),
            "within_median": float(np.median(x)),
            "between_median": float(np.median(y)),
            "p_value": p,
            "cliffs_delta": cliff_delta(y, x),
            "test": test,
        }
    )

stats_df = pd.DataFrame(stat_rows).sort_values("p_value")
stats_df.to_csv(OUT_DIR / "replicates_stats_within_vs_between.csv", index=False)

stats_df

fig_path


In [ ]:
stats_df